In [ ]:
from pathlib import Path

zip_path = Path("/content/faceShape.v4i.folder.zip")
print("ZIP found:", zip_path.is_file())

ZIP found: True


In [ ]:
from zipfile import ZipFile

data_dir = Path("/content/faceshape_v4")

with ZipFile(zip_path) as archive:
    archive.extractall(data_dir)

print("Extracted folders:", sorted(p.name for p in data_dir.iterdir() if p.is_dir()))

Extracted folders: ['test', 'train', 'valid']


In [ ]:
from collections import Counter

from torchvision.datasets import ImageFolder

test_data = ImageFolder(data_dir / "test")
counts = Counter(test_data.classes[label] for label in test_data.targets)

print("Test images:", len(test_data))
for label, count in sorted(counts.items()):
    print(f"{label}: {count}")

Test images: 440
heart: 198
long: 242


In [ ]:
from inference_sdk import InferenceConfiguration, InferenceHTTPClient

print("Roboflow SDK imported successfully")

Roboflow SDK imported successfully


In [ ]:
for split in ("train", "valid", "test"):
    data = ImageFolder(data_dir / split)
    label_counts = Counter(data.classes[i] for i in data.targets)
    print(split, dict(sorted(label_counts.items())))

train {'heart': 2315, 'long': 2688, 'oval': 2892, 'round': 3215, 'square': 3252}
valid {'heart': 2, 'long': 57, 'oval': 236, 'round': 235, 'square': 402}
test {'heart': 198, 'long': 242}


In [ ]:
from getpass import getpass

client = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key=getpass("Roboflow API key: ").strip(),
).configure(InferenceConfiguration(api_key_transport="header"))

Roboflow API key: ··········


In [ ]:
from random import Random

samples = list(test_data.samples)
Random(42).shuffle(samples)
seen = Counter()
pilot_results = []

for path, label_id in samples:
    actual = test_data.classes[label_id].lower()
    if seen[actual] >= 2:
        continue

    response = client.infer(path, model_id="faceshape-atkte/4")
    predicted = response["top"].lower()
    pilot_results.append((actual, predicted))
    seen[actual] += 1
    print(f"Actual: {actual} | Predicted: {predicted}")

correct = sum(actual == predicted for actual, predicted in pilot_results)
print(f"Pilot: {correct}/{len(pilot_results)} correct")

Actual: heart | Predicted: heart
Actual: long | Predicted: square
Actual: long | Predicted: long
Actual: heart | Predicted: heart
Pilot: 3/4 correct


In [ ]:
import json
from pathlib import Path

results_path = Path("/content/faceshape_v4_test_results.jsonl")
completed = set()

if results_path.exists():
    completed = {
        json.loads(line)["path"]
        for line in results_path.read_text().splitlines()
        if line.strip()
    }

with results_path.open("a", encoding="utf-8") as output:
    for path, label_id in test_data.samples:
        if path in completed:
            continue

        response = client.infer(path, model_id="faceshape-atkte/4")
        row = {
            "path": path,
            "actual": test_data.classes[label_id].lower(),
            "predicted": response["top"].lower(),
        }
        output.write(json.dumps(row) + "\n")
        output.flush()
        completed.add(path)

        if len(completed) % 25 == 0:
            print(f"Completed {len(completed)}/{len(test_data)}", flush=True)

print(f"Finished: {len(completed)}/{len(test_data)}")

Completed 25/440
Completed 50/440
Completed 75/440
Completed 100/440
Completed 125/440
Completed 150/440
Completed 175/440
Completed 200/440
Completed 225/440
Completed 250/440
Completed 275/440
Completed 300/440
Completed 325/440
Completed 350/440
Completed 375/440
Completed 400/440
Completed 425/440
Finished: 440/440


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

results = pd.read_json(results_path, lines=True)
assert len(results) == results["path"].nunique() == len(test_data)

actual = results["actual"]
predicted = results["predicted"]

print(f"Test accuracy: {accuracy_score(actual, predicted):.2%}")
print(classification_report(actual, predicted, labels=["heart", "long"],
                            digits=3, zero_division=0))

shapes = ["heart", "long", "oval", "round", "square"]
matrix = pd.crosstab(actual, predicted, rownames=["Actual"], colnames=["Predicted"])
display(matrix.reindex(index=["heart", "long"], columns=shapes, fill_value=0))

Test accuracy: 76.14%
              precision    recall  f1-score   support

       heart      0.920     0.818     0.866       198
        long      0.961     0.715     0.820       242

   micro avg      0.941     0.761     0.842       440
   macro avg      0.941     0.767     0.843       440
weighted avg      0.943     0.761     0.841       440



Predicted,heart,long,oval,round,square
Actual,,,,,
heart,162,7,16,1,12
long,14,173,18,3,33


In [ ]:
from google.colab import files

csv_path = "/content/roboflow_v4_heart_long_results.csv"
results.to_csv(csv_path, index=False)
files.download(csv_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from collections import Counter
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile

external_zip = Path("/content/external_test.zip")
external_dir = Path(mkdtemp(prefix="external_faces_", dir="/content"))
shapes = ["heart", "long", "oval", "round", "square"]
external_samples = []

with ZipFile(external_zip) as archive:
    for entry in archive.infolist():
        path = Path(entry.filename)
        if entry.is_dir() or "__MACOSX" in path.parts or path.name.startswith("."):
            continue
        if path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
            continue

        label = path.parent.name.strip().lower()
        if label in shapes:
            extracted_path = archive.extract(entry, external_dir)
            external_samples.append((Path(extracted_path), label))

counts = Counter(label for _, label in external_samples)
for shape in shapes:
    print(f"{shape}: {counts[shape]} photos")

print(f"Total: {len(external_samples)}")
assert all(counts[shape] > 0 for shape in shapes), "A class still has no photos."

heart: 198 photos
long: 242 photos
oval: 219 photos
round: 252 photos
square: 181 photos
Total: 1092


external_test/heart | .jpg | 198 files
external_test/long  | .jpg | 242 files
external_test/oval | .jpg | 219 files
external_test/round | .jpg | 252 files
external_test/square | .jpg | 181 files


In [ ]:
from collections import Counter, defaultdict
from hashlib import sha256
from pathlib import Path

def file_hash(path):
    return sha256(Path(path).read_bytes()).hexdigest()

original_splits = defaultdict(set)

for split in ("train", "valid", "test"):
    for path, _ in ImageFolder(data_dir / split).samples:
        original_splits[file_hash(path)].add(split)

overlap = Counter()

for path, label in external_samples:
    matches = original_splits.get(file_hash(path), set())
    source = ", ".join(sorted(matches)) if matches else "no exact match"
    overlap[label, source] += 1

for (label, source), count in sorted(overlap.items()):
    print(f"{label}: {count} | {source}")

heart: 198 | test
long: 242 | test
oval: 219 | train
round: 252 | train
square: 181 | train
